<center><h1>Batra_Raghav_HW3</h1></center>
<br>
<br>

Name: Raghav Batra
<br>
Github Username: rbatra02
<br>
USC ID: 3227519293

## 1. Time Series Classification Part 1: Feature Creation/Extraction

### (a) Download Data

Package imports

In [1]:
import pandas as pd, glob, os, re
import numpy as np
from scipy import stats

Get the AReM Data Set

In [2]:
DATA_DIR = "./AReM"

COLS = ["time", "avg_rss12", "var_rss12", "avg_rss13", "var_rss13", "avg_rss23", "var_rss23"]

def load_file(path):
    return pd.read_csv(path, comment="#", header=None, names=COLS, usecols=range(7), sep=r"[,\s]+", engine="python")

ACTIVITIES = ["bending1", "bending2", "cycling", "lying", "sitting", "standing", "walking"]

data = {}

for act in ACTIVITIES: 
    for path in glob.glob(os.path.join(DATA_DIR, act, "dataset*.csv")):
        n = int(re.search(r"dataset(\d+)", path).group(1))
        data[(act, n)] = load_file(path)


print(len(data))
print(sum(df.isna().sum().sum() for df in data.values()))
print({k: len(v) for k, v in data.items() if len(v) != 480})

0
0
{}


### (b) Test and Train Data

In [6]:
train, test = {}, {}

for(act, n), df in data.items():
    if act in ("bending1", "bending2"):
        test_ids = {1,2}
    else:
        test_ids = {1,2,3}
    if n in test_ids:
        test[(act, n)] = df
    else:
        train[(act,n)] = df

print(len(train), len(test))

69 19


### (c) Feature Extraction

#### i. Research

Statistical/Distribution
- min/max
- mean, median, mode
- standard deviation, variance
- range 
- first and third quartile, IQR
- percentiles
- skewness
- kurtosis

Energy/ magnitude
- root mean quare
- signal energy (sum of squared values)
- mean absolute value
- peak to peak amplitude

temporal structure
- autocorrelation
- zero crossing rate or mean crossing rate
- number of peaks
- slope or linear trend
- mean absolute difference between consecutive points

complexity
-entropy (sample entropy or approximate entropy)


#### ii. Extraction

In [11]:
SERIES = COLS[1:]
def extract_features(df):
    row = {}
    for i, col in enumerate(SERIES, start=1):
        s = df[col]
        row[f"min{i}"] = s.min()
        row[f"max{i}"] = s.max()
        row[f"mean{i}"] = s.mean()
        row[f"median{i}"] = s.median()
        row[f"std{i}"] = s.std()
        row[f"1st_quart{i}"] = s.quantile(0.25)
        row[f"3rd_quart{i}"] = s.quantile(0.75)
    return row

rows = []
for (act, n), df in sorted(data.items()):
    r = {"activity": act, "file": n, "split": "test" if (act,n) in test else "train"}
    r.update(extract_features(df))
    rows.append(r)

features = pd.DataFrame(rows)
features.index = range(1, len(features) + 1)
features.index.name = "Instance"

print(features.shape)
features.head()

feature_cols = features.columns[3:]
train_df = features[features.split == "train"]
test_df = features[features.split == "test"]

(88, 45)


#### iii. Standard Deviation

In [13]:
X = features[feature_cols].to_numpy()
n_rows = len(X)

rng = np.random.default_rng(42)
B = 1000
boot_stds = np.zeros((B, X.shape[1]))

for b in range(B):
    idx = rng.integers(0, n_rows, size=n_rows)
    boot_stds[b] = X[idx].std(axis=0, ddof=1)

ci = pd.DataFrame({
    "std": X.std(axis=0, ddof = 1),
    "CI_lower (5%)": np.percentile(boot_stds, 5, axis=0),
    "CI_upper (95%)": np.percentile(boot_stds, 95, axis=0),
}, index=feature_cols)

ci.round(3)

,std,CI_lower (5%),CI_upper (95%)
min1,9.570,8.210,10.787
max1,4.394,3.327,5.322
mean1,5.336,4.722,5.899
median1,5.440,4.785,6.003
std1,1.772,1.569,1.944
1st_quart1,6.154,5.597,6.665
3rd_quart1,5.139,4.334,5.843
min2,0.000,0.000,0.000
max2,5.063,4.645,5.398
mean2,1.574,1.394,1.702


#### iv. Select Features

In [17]:
train_df = features[features.split == "train"]
scores = {}
for ft in ["min", "max", "mean", "median", "std", "1st_quart", "3rd_quart"]:
    fs = []
    for k in range(1,7):
        col = f"{ft}{k}"
        if train_df[col].nunique() > 1:
            groups = [g[col] for _,g in train_df.groupby("activity")]
            fs.append(stats.f_oneway(*groups).statistic)
    scores[ft] = np.mean(fs)

pd.Series(scores).sort_values(ascending=False).round(1)

mean         368.2
3rd_quart    350.4
median       317.7
1st_quart    154.1
std          109.2
max           30.0
min           14.6
dtype: float64

mean, 3rd quartile and median have the largest f-scores.

## 2. ISLR 3.7.4

### (a) Linear Train

The cubic because it contains the linear model, its training RSS is always less than or equal the linear model's. The extra X^2 and X^3 terms will fit some of the random noise epsilon, which pushes training RSS a little lower.

### (b) Linear Test

The linear model because it already matches the true relationship, so it has no bias. The cubic adds nothing useful since its extra terms only fit noise which raises variance and causes overfitting. 

### (c) Not Linear Train

The cubic because it contains the linear model plus the extra X^2 and X^3 terms which capture some of the true nonlinear pattern, so its training RSS will be lower.

### (d) Not Linear Testing

Not enough information to tell as it depends how non linear the true relationship is. If the true relationship is very nonlinear, then the cubic model wins because the linear model has more bias. If the true relationship is only a little bit non linear then the linear model has small bias and may be better as the cubic model will have more variance.

## 3. ISLR 3.7.3 - Extra Practice 

## 3. ISLR 3.7.5 - Extra Practice 